In [1]:
from pathlib import Path
from eoflow.ea import EAWaterQualityAPI, get_ea_water_quality
from eoflow.utils import load_shapefile
import folium
import pandas as pd

In [2]:
try:
    file_path = Path(__file__).resolve()
except NameError:
    file_path = Path.cwd()

shapefile_path = file_path.parent.parent.parent.parent / "data/Counties_and_Unitary_Authorities_December_2023_Boundaries"
assert shapefile_path.is_dir()

In [3]:
gdf = load_shapefile(path=shapefile_path)
devon_gdf = gdf[gdf["CTYUA23NM"] == "Devon"]
devon_gdf

,CTYUA23CD,CTYUA23NM,CTYUA23NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
134,E10000008,Devon,None,283143,93085,-3.65698,50.7256,7aa8a35e-fa29-49bf-ac91-e4b121754500,"MULTIPOLYGON (((264838.577 43779.215, 264643.3..."


In [4]:

devon_gdf = devon_gdf.to_crs(epsg=4326)

# # Compute a sensible center for the map
geom_name = devon_gdf.geometry.name
centroid = devon_gdf.geometry.unary_union.centroid
map_center = [centroid.y, centroid.x]

# tooltip_fields = [c for c in gdf.columns if c != geom_name][:5]
# tooltip = folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_fields) if tooltip_fields else None

m = folium.Map(
    location=map_center, 
    zoom_start=9,
    width="400px",
    height="400px",
    scrollWheelZoom=False)

folium.GeoJson(
    devon_gdf.__geo_interface__,
    name="shapefile",
    style_function=lambda feat: {
        "fillColor": "#ff7800",
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.4,
    },
).add_to(m)

folium.LayerControl().add_to(m)

# Display the map
m

/tmp/ipykernel_356776/770127517.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = devon_gdf.geometry.unary_union.centroid


In [5]:
cs_df = pd.read_csv("/home/finley/Work/RDS/projects/enforce/data/CSI_Data/CSI_Data_ALL_12062025.csv")
cs_df.shape

(26152, 97)

In [6]:
cs_df["Date"] = pd.to_datetime(cs_df["Date"])

In [7]:
start_date = pd.to_datetime("2020-01-01")
end_date = pd.to_datetime("2020-06-01")

In [8]:
cs_df_ = cs_df[(cs_df["Date"] > start_date) & (cs_df["Date"] < end_date)]

In [10]:
from shapely.geometry import Point

# Get the Devon polygon
devon_polygon = devon_gdf.geometry.iloc[0]

# Filter cs_df_ to only include points inside the Devon polygon
def point_in_devon(row):
    try:
        if pd.isna(row['lat']) or pd.isna(row['long']):
            return False
        point = Point(row['long'], row['lat'])
        return devon_polygon.contains(point)
    except:
        return False

cs_df_devon = cs_df_[cs_df_.apply(point_in_devon, axis=1)].copy()

In [13]:
for _, row in cs_df_devon.iterrows():
    folium.Marker(
        location=[row["lat"], row["long"]],
        icon=folium.Icon(color="red") 
    ).add_to(m)

In [14]:
m

In [16]:
cs_df_devon.to_csv("cs_df_devon.csv")